## Overview

1. Clustering               pair shortlisting, Dinov2
2. Local matching           keypoints + matcher ALIKED/SuperPoint + LightGlue, or MASt3R
3. Geometric verification   RANSAC 
4. Reconstruction           COLMAP/pycolmap structure-from-motion (SfM) → camera poses
                            each separate reconstruction = one scene cluster; unregistered = outlier

## Pair shortlisting

Pair sortlisting via embedding using Dinov2
https://github.com/facebookresearch/dinov2

embed Images and group them into sets of similar images,
return a list of candidate pairs to match

In [62]:
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from pathlib import Path

dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
device = torch.device('cpu')

Using cache found in /Users/lucbaumeler/.cache/torch/hub/facebookresearch_dinov2_main


In [63]:
dinov2.eval().to(device)

transform = T.Compose([T.Resize((448, 448)), T.ToTensor(), T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])



In [64]:
x1 = Image.open('image-matching-challenge-2025/test/stairs/stairs_split_1_1710453576271.png').convert('RGB')
x2 = Image.open('image-matching-challenge-2025/test/stairs/stairs_split_1_1710453601885.png').convert('RGB')
x3 = Image.open('image-matching-challenge-2025/test/ETs/another_et_another_et001.png').convert('RGB')

x1 = transform(x1).unsqueeze(0).to(device)
x2 = transform(x2).unsqueeze(0).to(device)
x3 = transform(x3).unsqueeze(0).to(device)



with torch.no_grad():
    y1 = dinov2(x1)
    y2 = dinov2(x2)
    y3 = dinov2(x3)
    
sim = F.cosine_similarity(y1, y2)
sim2 = F.cosine_similarity(y1, y3)
print(sim.item())
print(sim2.item())


0.8893311023712158
0.02078670635819435


In [65]:
ds = ImageFolder('image-matching-challenge-2025/train', transform)
loader = DataLoader(dataset=ds, batch_size=64, shuffle=False)

embs = []
labls = []

with torch.no_grad():
    for imgs, labels in loader:
        embs.append(dinov2(imgs.to(device)))
        labls.append(labels.to(device))

In [66]:
print(embs[0])
print(labls[0])

tensor([[ 0.6501, -3.5805, -1.6924,  ...,  1.5323, -0.8058, -1.3643],
        [-0.9997, -3.7093, -0.3785,  ...,  0.5888,  0.0951, -1.5165],
        [-0.8266, -3.5939, -1.2035,  ...,  1.7604,  1.4490, -1.6209],
        ...,
        [ 2.5760, -0.2871, -1.6254,  ...,  0.5779,  2.3726,  3.6004],
        [ 1.4662, -0.4712, -2.8981,  ...,  1.2748,  3.1438,  3.2336],
        [ 1.3754, -0.6801, -1.5857,  ...,  2.2001,  3.8991,  2.8979]])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


In [67]:
E = torch.cat(embs)
L = torch.cat(labls)
emb = {name: E[L == idx] for name, idx in ds.class_to_idx.items()}
paths = {name: [p for p, l in ds.samples if l == idx] for name, idx in ds.class_to_idx.items()}

for name, X in emb.items():
    print(name, X.shape, len(paths[name]))

ETs torch.Size([22, 384]) 22
amy_gardens torch.Size([200, 384]) 200
fbk_vineyard torch.Size([163, 384]) 163
imc2023_haiper torch.Size([54, 384]) 54
imc2023_heritage torch.Size([209, 384]) 209
imc2023_theather_imc2024_church torch.Size([76, 384]) 76
imc2024_dioscuri_baalshamin torch.Size([138, 384]) 138
imc2024_lizard_pond torch.Size([214, 384]) 214
pt_brandenburg_british_buckingham torch.Size([225, 384]) 225
pt_piazzasanmarco_grandplace torch.Size([168, 384]) 168
pt_sacrecoeur_trevi_tajmahal torch.Size([225, 384]) 225
pt_stpeters_stpauls torch.Size([200, 384]) 200
stairs torch.Size([51, 384]) 51


In [77]:
from sklearn.cluster import HDBSCAN
X = F.normalize(emb['ETs'], dim=1).numpy()
labels = HDBSCAN(min_cluster_size=3, metric='cosine').fit_predict(X)

print(labels)

[ 0  0  0  0  0  0  0  0  0  0  2  2  2  2  2  1  1  1 -1 -1 -1 -1]


/opt/anaconda3/envs/imc2025/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
